In [2]:
import pandas as pd

# --- caminhos (o notebook está em notebooks/, os dados em dados/) ---
ped = pd.read_csv("../dados/pedidos.csv")
tax = pd.read_csv("../dados/taxas_vigentes.csv")

# converter data para tipo data
ped["data"] = pd.to_datetime(ped["data"])
tax["data"] = pd.to_datetime(tax["data"])

print("PEDIDOS:", ped.shape)
print(ped.dtypes)
ped.head()

PEDIDOS: (463114, 6)
pedido_id               int64
data           datetime64[us]
categoria                 str
macro_grupo               str
itens                   int64
gmv                   float64
dtype: object


,pedido_id,data,categoria,macro_grupo,itens,gmv
0,1129122,2025-08-01,CAT-17,Mídia & Papelaria,1,10.95
1,935870,2025-08-01,CAT-36,Eletro & Tech,1,3.43
2,987060,2025-08-01,CAT-17,Mídia & Papelaria,1,25.81
3,1073838,2025-08-01,CAT-09,Casa & Construção,1,29.36
4,1014023,2025-08-01,CAT-08,Saúde & Beleza,1,16.75


In [3]:
# ===== FASE 0 — checagem de saúde da base =====

# 1. período coberto
print("PERÍODO pedidos:", ped["data"].min().date(), "->", ped["data"].max().date())
print("PERÍODO taxas:  ", tax["data"].min().date(), "->", tax["data"].max().date())

# 2. categorias batem nos dois arquivos?
print("\nCategorias em pedidos:", ped["categoria"].nunique())
print("Categorias em taxas:  ", tax["categoria"].nunique())

# 3. alguma categoria troca de macro_grupo? (deve ser 0)
troca = ped.groupby("categoria")["macro_grupo"].nunique()
print("Categorias em >1 macro_grupo:", (troca > 1).sum())

# 4. toda venda acha sua taxa? (linhas órfãs — deve ser 0)
tax_keys = set(zip(tax["data"], tax["categoria"]))
orfas = sum(1 for k in zip(ped["data"], ped["categoria"]) if k not in tax_keys)
print("\nVendas SEM taxa correspondente:", orfas)

# 5. taxas duplicadas? (deve ser 0)
dups = tax.duplicated(subset=["data", "categoria"]).sum()
print("Taxas duplicadas (data+categoria):", dups)

# 6. gmv com problema? (zerado ou negativo)
print("\nGMV <= 0:", (ped["gmv"] <= 0).sum())
print("GMV nulo:", ped["gmv"].isna().sum())

PERÍODO pedidos: 2025-08-01 -> 2026-08-31
PERÍODO taxas:   2025-08-01 -> 2026-08-31

Categorias em pedidos: 49
Categorias em taxas:   49
Categorias em >1 macro_grupo: 0

Vendas SEM taxa correspondente: 0
Taxas duplicadas (data+categoria): 0

GMV <= 0: 0
GMV nulo: 0


In [4]:
# ===== FASE 1 — base fato (fonte da verdade) =====

# 1. cruzar cada venda com a taxa do seu dia+categoria
fato = ped.merge(tax, on=["data", "categoria"], how="left")

# 2. colunas de dinheiro
fato["comissao_rs"] = fato["gmv"] * fato["taxa_comissao_pct"] / 100
fato["cashback_rs"] = fato["gmv"] * fato["taxa_cashback_pct"] / 100
fato["net_take_pp"] = fato["taxa_comissao_pct"] - fato["taxa_cashback_pct"]
fato["margem_rs"]   = fato["comissao_rs"] - fato["cashback_rs"]

# 3. coluna de tempo (mês)
fato["ano_mes"] = fato["data"].dt.to_period("M").astype(str)

# 4. conferências: o merge manteve o número de linhas? sobrou taxa vazia?
print("Linhas após merge:", fato.shape[0], "(esperado 463114)")
print("Taxas não encontradas no merge:", fato["taxa_comissao_pct"].isna().sum(), "(esperado 0)")
print()
fato.head()

Linhas após merge: 463114 (esperado 463114)
Taxas não encontradas no merge: 0 (esperado 0)



,pedido_id,data,categoria,macro_grupo,itens,gmv,taxa_comissao_pct,taxa_cashback_pct,comissao_rs,cashback_rs,net_take_pp,margem_rs,ano_mes
0,1129122,2025-08-01,CAT-17,Mídia & Papelaria,1,10.95,5.0,1.0,0.5475,0.1095,4.0,0.4380,2025-08
1,935870,2025-08-01,CAT-36,Eletro & Tech,1,3.43,4.0,1.0,0.1372,0.0343,3.0,0.1029,2025-08
2,987060,2025-08-01,CAT-17,Mídia & Papelaria,1,25.81,5.0,1.0,1.2905,0.2581,4.0,1.0324,2025-08
3,1073838,2025-08-01,CAT-09,Casa & Construção,1,29.36,6.0,1.0,1.7616,0.2936,5.0,1.4680,2025-08
4,1014023,2025-08-01,CAT-08,Saúde & Beleza,1,16.75,9.0,3.0,1.5075,0.5025,6.0,1.0050,2025-08


In [5]:
# salvar a base fato para os materiais de apoio
fato.to_csv("../saidas/base_fato.csv", index=False)
print("Base fato salva em saidas/base_fato.csv")

Base fato salva em saidas/base_fato.csv


In [6]:
# ===== FASE 2 — preparar série diária por categoria =====

# agregação diária: para cada dia + categoria, somamos o dinheiro
# e pegamos o cashback do dia (é igual para toda a categoria naquele dia)
serie_diaria = (
    fato.groupby(["data", "categoria", "macro_grupo"])
    .agg(
        gmv=("gmv", "sum"),
        margem_rs=("margem_rs", "sum"),
        cashback_pct=("taxa_cashback_pct", "first"),
        comissao_pct=("taxa_comissao_pct", "first"),
    )
    .reset_index()
)

print("Série diária:", serie_diaria.shape)
print("Categorias disponíveis:", sorted(serie_diaria["categoria"].unique()))
serie_diaria.head()

Série diária: (14365, 7)
Categorias disponíveis: ['CAT-01', 'CAT-02', 'CAT-03', 'CAT-04', 'CAT-05', 'CAT-06', 'CAT-07', 'CAT-08', 'CAT-09', 'CAT-10', 'CAT-11', 'CAT-12', 'CAT-13', 'CAT-14', 'CAT-15', 'CAT-16', 'CAT-17', 'CAT-18', 'CAT-19', 'CAT-20', 'CAT-21', 'CAT-22', 'CAT-23', 'CAT-24', 'CAT-25', 'CAT-26', 'CAT-27', 'CAT-28', 'CAT-29', 'CAT-30', 'CAT-31', 'CAT-32', 'CAT-33', 'CAT-34', 'CAT-35', 'CAT-36', 'CAT-37', 'CAT-38', 'CAT-39', 'CAT-40', 'CAT-41', 'CAT-42', 'CAT-43', 'CAT-44', 'CAT-45', 'CAT-46', 'CAT-47', 'CAT-48', 'CAT-49']


,data,categoria,macro_grupo,gmv,margem_rs,cashback_pct,comissao_pct
0,2025-08-01,CAT-02,Alimentos & Bebidas,324.66,16.2330,1.0,6.0
1,2025-08-01,CAT-03,Casa & Construção,398.24,19.9120,1.0,6.0
2,2025-08-01,CAT-04,Moda & Acessórios,341.87,20.5122,1.0,7.0
3,2025-08-01,CAT-05,Eletro & Tech,10702.95,321.0885,1.0,4.0
4,2025-08-01,CAT-07,Esporte & Lazer,237.86,14.2716,1.0,7.0


In [7]:
# ===== FASE 2 — explorador interativo (Plotly) =====
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def explorar(categorias, freq="D"):
    """
    categorias: lista de códigos, ex ["CAT-08", "CAT-17"]
    freq: "D" = diário | "M" = mensal (visão macro)
    """
    if isinstance(categorias, str):
        categorias = [categorias]

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=("Cashback (%)", "GMV (R$)", "Margem (R$)"),
    )

    for cat in categorias:
        d = serie_diaria[serie_diaria["categoria"] == cat].copy().sort_values("data")

        if freq == "M":
            # visão macro: agrega por mês. GMV/margem somam, cashback vira média do mês
            g = d.set_index("data").resample("MS").agg(
                gmv=("gmv", "sum"),
                margem_rs=("margem_rs", "sum"),
                cashback_pct=("cashback_pct", "mean"),
            ).reset_index()
            x = g["data"]
            cb, gmv, mg = g["cashback_pct"], g["gmv"], g["margem_rs"]
        else:
            x = d["data"]
            cb, gmv, mg = d["cashback_pct"], d["gmv"], d["margem_rs"]

        fig.add_trace(go.Scatter(x=x, y=cb, name=cat, legendgroup=cat,
                                 mode="lines"), row=1, col=1)
        fig.add_trace(go.Scatter(x=x, y=gmv, name=cat, legendgroup=cat,
                                 showlegend=False, mode="lines"), row=2, col=1)
        fig.add_trace(go.Scatter(x=x, y=mg, name=cat, legendgroup=cat,
                                 showlegend=False, mode="lines"), row=3, col=1)

    fig.update_layout(height=700, hovermode="x unified",
                      title=f"Explorador — {', '.join(categorias)} ({'mensal' if freq=='M' else 'diário'})")
    fig.show()

# --- exemplo: rode e depois troque as categorias ---
explorar(["CAT-08"], freq="D")

In [8]:
# ===== FASE 2 — explorador com dropdown (ipywidgets) =====
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cats_disponiveis = sorted(serie_diaria["categoria"].unique())
grupos_disponiveis = sorted(serie_diaria["macro_grupo"].unique())

# --- controles ---
sel_grupo = widgets.Dropdown(
    options=["(todos)"] + grupos_disponiveis,
    value="(todos)", description="Grupo:",
)
sel_cats = widgets.SelectMultiple(
    options=cats_disponiveis, value=("CAT-08",),
    description="Categorias:", rows=8,
)
sel_freq = widgets.ToggleButtons(
    options=[("Diário", "D"), ("Mensal", "M")],
    value="D", description="Visão:",
)
saida = widgets.Output()

# quando muda o grupo, filtra as categorias daquele grupo na lista
def ao_mudar_grupo(change):
    if sel_grupo.value == "(todos)":
        sel_cats.options = cats_disponiveis
    else:
        cats_do_grupo = sorted(
            serie_diaria[serie_diaria["macro_grupo"] == sel_grupo.value]["categoria"].unique()
        )
        sel_cats.options = cats_do_grupo
sel_grupo.observe(ao_mudar_grupo, names="value")

def desenhar(change=None):
    with saida:
        saida.clear_output(wait=True)
        categorias = list(sel_cats.value)
        if not categorias:
            print("Selecione ao menos uma categoria.")
            return
        freq = sel_freq.value

        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                            subplot_titles=("Cashback (%)", "GMV (R$)", "Margem (R$)"))
        for cat in categorias:
            d = serie_diaria[serie_diaria["categoria"] == cat].copy().sort_values("data")
            if freq == "M":
                g = d.set_index("data").resample("MS").agg(
                    gmv=("gmv","sum"), margem_rs=("margem_rs","sum"),
                    cashback_pct=("cashback_pct","mean")).reset_index()
                x, cb, gmv, mg = g["data"], g["cashback_pct"], g["gmv"], g["margem_rs"]
            else:
                x, cb, gmv, mg = d["data"], d["cashback_pct"], d["gmv"], d["margem_rs"]
            fig.add_trace(go.Scatter(x=x, y=cb, name=cat, legendgroup=cat, mode="lines"), row=1, col=1)
            fig.add_trace(go.Scatter(x=x, y=gmv, name=cat, legendgroup=cat, showlegend=False, mode="lines"), row=2, col=1)
            fig.add_trace(go.Scatter(x=x, y=mg, name=cat, legendgroup=cat, showlegend=False, mode="lines"), row=3, col=1)
        fig.update_layout(height=700, hovermode="x unified",
                          title=f"{', '.join(categorias)} ({'mensal' if freq=='M' else 'diário'})")
        fig.show()

# liga os controles ao redesenho
sel_cats.observe(desenhar, names="value")
sel_freq.observe(desenhar, names="value")

display(widgets.HBox([sel_grupo, sel_freq]), sel_cats, saida)
desenhar()

SelectMultiple(description='Categorias:', index=(7,), options=('CAT-01', 'CAT-02', 'CAT-03', 'CAT-04', 'CAT-05…

Output()

In [ ]:
# ===== FASE 2 — explorador estilizado =====
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cats_disponiveis = sorted(serie_diaria["categoria"].unique())
grupos_disponiveis = sorted(serie_diaria["macro_grupo"].unique())

# paleta agradável (contrastante, sem serrilhar a vista)
PALETA = ["#6C5CE7", "#00B894", "#E17055", "#0984E3", "#FDCB6E",
          "#E84393", "#00CEC9", "#A29BFE", "#FAB1A0", "#55EFC4"]

sel_grupo = widgets.Dropdown(options=["(todos)"] + grupos_disponiveis, value="(todos)", description="Grupo:")
sel_cats  = widgets.SelectMultiple(options=cats_disponiveis, value=("CAT-08",), description="Categorias:", rows=8)
sel_freq  = widgets.ToggleButtons(options=[("Diário","D"),("Mensal","M")], value="D", description="Visão:")
saida = widgets.Output()

def ao_mudar_grupo(change):
    if sel_grupo.value == "(todos)":
        sel_cats.options = cats_disponiveis
    else:
        sel_cats.options = sorted(serie_diaria[serie_diaria["macro_grupo"]==sel_grupo.value]["categoria"].unique())
sel_grupo.observe(ao_mudar_grupo, names="value")

def desenhar(change=None):
    with saida:
        saida.clear_output(wait=True)
        categorias = list(sel_cats.value)
        if not categorias:
            print("Selecione ao menos uma categoria."); return
        freq = sel_freq.value

        fig = make_subplots(
            rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.09,
            subplot_titles=("<b>Cashback</b> (%)", "<b>GMV</b> (R$)", "<b>Margem</b> (R$)"),
        )

        for i, cat in enumerate(categorias):
            cor = PALETA[i % len(PALETA)]
            d = serie_diaria[serie_diaria["categoria"]==cat].copy().sort_values("data")
            if freq == "M":
                g = d.set_index("data").resample("MS").agg(
                    gmv=("gmv","sum"), margem_rs=("margem_rs","sum"),
                    cashback_pct=("cashback_pct","mean")).reset_index()
                x, cb, gmv, mg = g["data"], g["cashback_pct"], g["gmv"], g["margem_rs"]
            else:
                x, cb, gmv, mg = d["data"], d["cashback_pct"], d["gmv"], d["margem_rs"]

            linha = dict(color=cor, width=2, shape="spline", smoothing=0.6)
            fig.add_trace(go.Scatter(x=x, y=cb, name=cat, legendgroup=cat, mode="lines", line=linha), row=1, col=1)
            fig.add_trace(go.Scatter(x=x, y=gmv, name=cat, legendgroup=cat, showlegend=False, mode="lines", line=linha), row=2, col=1)
            fig.add_trace(go.Scatter(x=x, y=mg, name=cat, legendgroup=cat, showlegend=False, mode="lines", line=linha), row=3, col=1)

        # eixo do tempo: 1 marca por mês, formato "Set/25"
        fig.update_xaxes(
            dtick="M1", tickformat="%b/%y", ticklabelmode="period",
            showgrid=True, gridcolor="rgba(255,255,255,0.06)", ticks="outside",
            tickangle=0, tickfont=dict(size=11),
        )
        fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.06)", zeroline=False)

        fig.update_layout(
            template="plotly_dark",
            height=720,
            hovermode="x unified",
            title=dict(text=f"<b>{', '.join(categorias)}</b>  ·  visão {'mensal' if freq=='M' else 'diária'}",
                       x=0.5, font=dict(size=18)),
            plot_bgcolor="#1e1e2e", paper_bgcolor="#1e1e2e",
            font=dict(family="Segoe UI, Arial", color="#e0e0e0", size=12),
            legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="center", x=0.5,
                        bgcolor="rgba(0,0,0,0)"),
            margin=dict(l=60, r=30, t=90, b=40),
        )
        for ann in fig.layout.annotations:  # títulos dos painéis
            ann.font = dict(size=13, color="#b0b0c0")
        fig.show()

sel_cats.observe(desenhar, names="value")
sel_freq.observe(desenhar, names="value")

display(widgets.HBox([sel_grupo, sel_freq]), sel_cats, saida)
desenhar()

SelectMultiple(description='Categorias:', index=(7,), options=('CAT-01', 'CAT-02', 'CAT-03', 'CAT-04', 'CAT-05…

Output()